# JSE Top 40: three-cross-section study

This walkthrough extends the original single-window SACJ study to **30 June 2025, 31 December 2025 and 30 June 2026**. It starts with EDA, compares models, explains cluster membership with **SHAP**, and compares continuing members across dates.

There are **41 instruments per date**, 123 instrument-date records and 43 distinct tickers. These are overlapping index cross-sections, not 123 independent firms. The task is descriptive investment screening, not strategy-return prediction.

**Before running:** select the project `.venv` Python kernel in VS Code. Install `requirements-notebook.txt` in that environment. Then choose **Run All**. A new immutable run folder is created automatically. The command-line equivalent is `python scripts/run_study.py`.

Original inputs and evidence flags stay intact. RNI remains with separately flagged model-only imputations. The recovered t-SNE baseline and a frozen UMAP sensitivity are displayed separately; nothing is automatically promoted to the paper.

## 1. Setup and validate the historical inputs

The implementation lives in short modules under `scripts/`. This notebook calls the same functions as the command-line workflow, so it does not contain a second, conflicting analysis implementation.

Keep the defaults for the first run. Five distinct seeds measure sensitivity; they are not bootstrap confidence intervals. The UMAP sensitivity uses the previously explored June settings (3 neighbours, min_dist=0.01, k=8), frozen across dates.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "scripts").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "scripts" / "study_workflow.py").is_file():
    raise RuntimeError("Open this notebook from the JSE-Top-40-Clustering repository.")
sys.path.insert(0, str(ROOT / "scripts"))

from IPython.display import display
from study_workflow import StudySession

SEED = 42
STABILITY_RUNS = 5
INCLUDE_UMAP_SENSITIVITY = True
RUN_ID = os.environ.get("JSE_STUDY_RUN_ID")  # None gives a unique run ID

study = StudySession(run_id=RUN_ID, seed=SEED, stability_runs=STABILITY_RUNS,
                     include_umap=INCLUDE_UMAP_SENSITIVITY)
data = study.load_data()
display(data[["decision_date", "ticker", "company", "sector_label"]].head())
print("The ten numeric model features are:", study.features)

## 2. Exploratory data analysis (EDA)

Inspect the population, raw missing cases, model-only imputations, distributions and correlations **before fitting clusters**. June 2025 supplies the fixed imputation/clipping/scaling information. Later MinMax-transformed values can exceed [0,1]. Sector labels are retained as context, not numeric model features.

Raw financial definitions, ZAc/ZAR conventions and reconstruction limitations are documented in `docs/orion_methods_results_checklist.md`. The plots include flagged model-only imputations; those values are not reported issuer fundamentals.

In [ ]:
eda = study.explore()
display(eda["population"])
display(eda["missingness"].query("raw_missing > 0"))
display(eda["evidence"])
print("Distinct tickers:", data.ticker.nunique())
print("Observed at all three dates:", int(eda["presence"].all(axis=1).sum()))

In [ ]:
study.show_figure("eda/population_and_sectors.png")
study.show_figure("eda/raw_missingness.png")
study.show_figure("eda/feature_distributions.png")

### Membership and correlations

Membership absence is not a negative screening flag. The adjacent date pairs each have 40 continuing instruments; only 39 are observed at all three dates.

Correlated size features can affect clustering and surrogate importance. The table flags |Pearson r| >= 0.8 descriptively; it does not silently remove features from the documented baseline or test a new feature set.

In [ ]:
study.show_figure("eda/membership_presence.png")
display(eda["correlated_pairs"])
for date in study.config["dates"]:
    study.show_figure(f"eda/correlation_{date}.png")

## 3. Modelling: the same workflow at each date

The recovered baseline is MinMax -> 2D t-SNE (perplexity 5, learning rate 10, 300 iterations) -> average-linkage Manhattan hierarchical clustering, k=8. The comparison includes PCA/t-SNE/UMAP with hierarchical clustering, K-Means and DBSCAN on **each date**.

The extra UMAP case is a frozen sensitivity, not an automatic replacement for the baseline. No later-date score selects its parameters. Each representation and clustering is independently refitted at each date.

Report **silhouette (higher better)** and **Davies-Bouldin (lower better)** together. The scores use Euclidean geometry in the embedding and common financial-feature spaces. Linkage uses Manhattan distance. DBSCAN scores exclude noise and must be read with coverage.

**k=8 is fixed by the original eight-sector design for hierarchical clustering and K-Means.** The k-sweep is diagnostic only and does not replace this choice. DBSCAN retains its own cluster/noise count. The sector categories remain a contextual comparator, not a fit target or independently certified historic ICB labels.

In [ ]:
model_comparison = study.fit_models()
score_columns = ["decision_date", "model", "n_clusters", "coverage",
                 "silhouette_embedding", "davies_bouldin_embedding",
                 "silhouette_features", "davies_bouldin_features"]
display(model_comparison[score_columns].round(4))
for date in study.config["dates"]:
    study.show_figure(f"model_quality_{date}.png")

### Cluster maps, feature profiles and dendrograms

Change `VIEW_DATE` to inspect another snapshot without refitting models. Each map is an independent fit: axes and raw cluster numbers are not comparable economic coordinates/identities across dates. Membership lists keep crowded tickers readable; * marks a model-only imputation.

The median-deviation heatmap compares each cluster with its same-date population median in the common June-scaled feature space. It is a descriptive profile, not a significance test.

In [ ]:
VIEW_DATE = "2025-06-30"  # also "2025-12-31" or "2026-06-30"
for model_name, folder in study.model_folders.items():
    relative_folder = folder.relative_to(study.out)
    print(model_name, VIEW_DATE)
    study.show_figure(relative_folder / VIEW_DATE / "cluster_map.png")
    study.show_figure(relative_folder / VIEW_DATE / "feature_profile.png")
    study.show_figure(relative_folder / VIEW_DATE / "dendrogram.png")

### Seed and cluster-count sensitivity

Inspect all pairwise ARI/NMI values, not only a favourable seed. The representative model/SHAP figures use seed 42. Sector ARI/NMI in the comparison are contextual agreement statistics; the legacy sectors are not a tuning target or certified historical ICB ground truth.

The k-sensitivity figures show both score geometries at every date. The dashed line is the fixed specification, not a newly selected optimum at each date.

In [ ]:
display(study.seed_agreement.groupby(["study_model", "decision_date"])
        .agg(pairs=("ari", "size"), mean_ari=("ari", "mean"), min_ari=("ari", "min"),
             mean_nmi=("nmi", "mean")).round(4))
for model_name, folder in study.model_folders.items():
    study.show_figure(folder.relative_to(study.out) / "cluster_count_sensitivity.png")

## 4. Explain cluster membership with SHAP

A gradient-boosted XGBoost surrogate learns each discovered partition from the original ten financial features. **SHAP explains that surrogate**, not the clustering algorithm directly. It describes associations, not causality, valuation or future returns.

Report in-sample fidelity separately from stratified out-of-fold (OOF) fidelity. If a cluster is a singleton, OOF is explicitly not estimable. OOF fidelity to full-snapshot pseudo-labels does not validate the cluster discovery itself. Class mapping, feature order and raw-margin SHAP additivity are checked and saved.

In [ ]:
surrogate_fidelity = study.explain_clusters()
display(surrogate_fidelity.round(6))

### Read the explanation plots

The global bar chart averages absolute SHAP over firms/classes. The distribution plot is an example for the **largest local class**: positive SHAP increases its surrogate raw margin. Colour shows relative feature values within each feature for display only. Per-cluster bars, raw feature values, classes, base values and SHAP arrays are also saved.

The exported `shap_univariate_comparison.csv` compares own-class SHAP rankings with univariate cluster median deviations. Neither is a causal explanation. Singleton subgroup inference remains unreliable.

In [ ]:
for model_name, folder in study.model_folders.items():
    relative_folder = folder.relative_to(study.out)
    print(model_name, VIEW_DATE)
    study.show_figure(relative_folder / VIEW_DATE / "shap_global.png")
    study.show_figure(relative_folder / VIEW_DATE / "shap_distribution.png")
    display(study.table(relative_folder / VIEW_DATE / "shap_univariate_rank_summary.csv").round(4))

## 5. Compare dates while respecting index membership

ARI/NMI compare continuing members only. Hungarian maximum-overlap alignment creates descriptive transition tables; labels do not become persistent economic identities. Entries/exits are displayed separately and receive **no ordinary cluster-change flag**. Imputation-sensitive interpretations remain marked.

The separate Jaccard-alignment sensitivity is documented in `reports/orion_alignment_differences.csv` and the ORiON checklist; alignment objectives can change instrument-level interpretations. No emerging/persistent/resolving anomaly taxonomy is invented here.

In [ ]:
temporal_agreement = study.compare_dates()
display(temporal_agreement.round(4))
for model_name, folder in study.model_folders.items():
    relative_folder = folder.relative_to(study.out)
    study.show_figure(relative_folder / "transition_heatmaps.png")
    events = study.table(relative_folder / "membership_and_cluster_transitions.csv")
    display(events.loc[events.membership_event != "continuing",
                       ["before", "after", "ticker", "membership_event", "cluster_changed"]])

## 6. Validate and save the complete study

The run is marked completed only after EDA, models, plots, all required SHAP outputs, temporal comparisons and artifact checks succeed. A unique run ID prevents overwriting earlier work. The registry records configuration, code/data versions, environment and status.

If execution stops before this cell, the run is incomplete (or failed), not a completed result. Start a new run rather than reusing its ID. The source notebook remains clean; an executed copy may be saved separately under `reports/`.

In [ ]:
completed_folder = study.finish()
display(study.table("primary_model_summary.csv")[[
    "study_model", "decision_date", "silhouette_embedding", "davies_bouldin_embedding",
    "silhouette_features", "davies_bouldin_features", "mean_seed_ari",
    "in_sample_accuracy", "oof_accuracy"]].round(4))
print("Read:", completed_folder / "study_summary.md")
print("Verify with: python scripts/verify_run.py", completed_folder)

## What the revised paper can claim

Three repeated cross-sections extend the empirical scope and allow descriptive temporal checks beyond the original single window. They do not create 123 independent firms or establish investment usefulness by themselves.

The ten-feature historic-consensus-excluded variant differs from the original eleven-feature experiment. Archived shares/capitalisation, historical sector and daily market-source evidence remain unresolved. No output is certified investor-observable PIT solely because the code runs. The baseline pointer remains unpromoted. Hotelling calibration and temporal anomaly definitions are separate methodological decisions.